In [15]:
!pip install -q pymupdf transformers torch

In [16]:
import fitz
import torch
import re
import requests
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

Using device: GPU


In [22]:
data1 = """
Bangladesh is a beautiful country in South Asia, known for its rich cultural heritage, fertile land,
and vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political
center but also the economic and cultural heart of the nation. It is one of the most densely
populated cities in the world and plays a major role in the country’s development.
One of the most important natural features of Bangladesh is the Padma River, which is often
referred to as the lifeline of the country. This river is crucial for transportation, agriculture,
fishing, and daily life. Many people depend on it for their livelihoods, and it supports the fertile
land that makes Bangladesh suitable for farming.
Bangladesh gained its independence in 1971. This historic event is remembered every year as
Independence Day and is a source of great pride for the people of Bangladesh. The struggle for
independence played a key role in shaping the nation’s identity and unity.
The official language of Bangladesh is Bengali, also known as Bangla. It is spoken by the
majority of the population and is deeply connected to the country’s culture, literature, and
traditions. The importance of the language is highlighted by International Mother Language Day,
which originated from the language movement in Bangladesh.
Bangladesh is also famous for the Sundarbans, the largest mangrove forest in the world. This
forest is a UNESCO World Heritage Site and is known for its rich biodiversity. It is home to many
species of animals, including the famous Royal Bengal Tiger. The Sundarbans also play an
important role in protecting the coastal areas from natural disasters like cyclones.
The official currency of Bangladesh is the Bangladeshi Taka. It is used in all types of financial
transactions across the country. The economy of Bangladesh has been growing steadily, with
key sectors including agriculture, textiles, and remittances from abroad.
The national animal of Bangladesh is the Royal Bengal Tiger. This majestic animal represents
strength, courage, and pride. It is mainly found in the Sundarbans and is an important symbol of
the country’s wildlife heritage.
In conclusion, Bangladesh is a country full of natural beauty, historical significance, and cultural
richness. From its vibrant capital city to its mighty rivers and unique forests, it offers a diverse
and fascinating identity that continues to grow and develop over time."""

data2 = """
Bangladesh(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&
&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&
&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%
%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%
%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$
$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((
((((((_. is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and
vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political
center but also the economic and cultural heart of the nation. It is one of the most densely
populated cities in the world and plays a major role in the country’s development.
One of the most important natural features of Bangladesh is the
(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&()
)))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&
&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$
$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%
%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$
%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_.
Padma River, which is often referred to as the lifeline of the country. This river is crucial for
transportation, agriculture,
fishing, and daily life. Many people depend on it for their livelihoods, and it supports the fertile
land that makes Bangladesh suitable for farming.
Bangladesh gained its independence in 1971. This historic event is remembered every year as
Independence Day and is a source of great pride
for the people of Bangladesh. The struggle for independence played a key role in shaping the
nation’s identity and unity.
The official language of Bangladesh is Bengali, also known as Bangla. It is spoken by the
majority of the population and is deeply connected to the country’s culture, literature, and
traditions. The importance of the language is highlighted by International Mother Language Day,
which originated from the language movement in Bangladesh.
(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&()
)))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&
&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$
$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%
%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$
%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_.

Bangladesh is also famous for the Sundarbans, the largest mangrove forest in the world. This
forest is a UNESCO World Heritage Site and is known for its rich biodiversity. It is home to many
species of animals, including the
$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&()))
)))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&
&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$
$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%
%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$
%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$
$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))
(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&
&&&&&&())))))))))(((((((((((((((_.
The official currency of Bangladesh is the Bangladeshi Taka. It is used in all types of financial
transactions across the country.
(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&()
)))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&
&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$
$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%
%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$
%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_. The
economy of Bangladesh has been growing steadily, with key sectors including agriculture,
textiles, and remittances from abroad.
The national animal of Bangladesh is the Royal Bengal Tiger. This majestic animal represents
strength, courage, and pride. It is
mainly found in the Sundarbans and is an important symbol of the country’s wildlife heritage.
In conclusion, Bangladesh is a country full of natural beauty, historical significance, and cultural
richness. From its vibrant capital city to its
mighty rivers
and unique forests, it offers a diverse and fascinating identity that continues to grow and
develop over time."""

full_text1 = re.sub(r'\s+', ' ', data1).strip()
full_text2 = re.sub(r'\s+', ' ', data2).strip()

In [23]:
def get_chunks(text, chunk_size=1000, overlap=200):
    """Splits text into overlapping chunks to handle model token limits."""
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i + chunk_size])
    return chunks

chunks1 = get_chunks(full_text1)
chunks2 = get_chunks(full_text2)

In [24]:
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad", device=device)

def ask_question(question, context_chunks):
    best_answer = {"score": 0, "answer": "I couldn't find an answer in the document."}

    for chunk in context_chunks:
        result = qa_pipeline(question=question, context=chunk)

        if result['score'] > best_answer['score']:
            best_answer = result

    return best_answer['answer'], best_answer['score']

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [28]:
user_query = ["What is the capital city of Bangladesh?", "Which river is considered the lifeline of Bangladesh?",
              "In which year did Bangladesh gain independence?", "What is the official language of Bangladesh?",
              "What is the name of the world’s largest mangrove forest located in Bangladesh?",
              "Which currency is used in Bangladesh?", "What is the national animal of Bangladesh?"]

for i in user_query:
    answer, score = ask_question(i, chunks1)
    print(f"\nQuestion: {i}")
    print(f"Answer: {answer}")

print("\nGAP")

for j in user_query:
    answer, score = ask_question(j, chunks2)
    print(f"\nQuestion: {j}")
    print(f"Answer: {answer}")


Question: What is the capital city of Bangladesh?
Answer: Dhaka

Question: Which river is considered the lifeline of Bangladesh?
Answer: Padma River

Question: In which year did Bangladesh gain independence?
Answer: 1971

Question: What is the official language of Bangladesh?
Answer: Bengali

Question: What is the name of the world’s largest mangrove forest located in Bangladesh?
Answer: Sundarbans

Question: Which currency is used in Bangladesh?
Answer: ntinues

Question: What is the national animal of Bangladesh?
Answer: Royal Bengal Tiger

GAP

Question: What is the capital city of Bangladesh?
Answer: Dhaka

Question: Which river is considered the lifeline of Bangladesh?
Answer: Bangladeshi Taka

Question: In which year did Bangladesh gain independence?
Answer: 1971

Question: What is the official language of Bangladesh?
Answer: Bangladeshi Taka

Question: What is the name of the world’s largest mangrove forest located in Bangladesh?
Answer: Sundarbans

Question: Which currency is 